# Support Vector Machines from scratch 
1. Math/math.md worked example (margin, support vectors)
2. Linear SVM via hinge-loss (sub)gradient descent
3. Effect of C
4. Kernels (linear / polynomial / RBF) and a simplified SMO dual solver
5. Kernel trick demo on circular data, gamma effect
6. Hinge vs logistic loss (how many points influence the model)
7. Support Vector Regression (epsilon-insensitive loss)

In [ ]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

os.makedirs('plots', exist_ok=True)
rng = np.random.default_rng(0)

def standardize(X, mu=None, sd=None):
    """Always scale features before an SVM."""
    mu = X.mean(0) if mu is None else mu
    sd = X.std(0) + 1e-12 if sd is None else sd
    return (X - mu) / sd, mu, sd

def split(X, y, frac=0.3, seed=0):
    r = np.random.default_rng(seed); p = r.permutation(len(y)); k = int(len(y) * frac)
    return X[p[k:]], X[p[:k]], y[p[k:]], y[p[:k]]

def plot_boundary(ax, predict, X, y, title, decision=None, levels=(-1, 0, 1)):
    gx, gy = np.meshgrid(np.linspace(X[:, 0].min()-.5, X[:, 0].max()+.5, 200),
                         np.linspace(X[:, 1].min()-.5, X[:, 1].max()+.5, 200))
    G = np.c_[gx.ravel(), gy.ravel()]
    ax.contourf(gx, gy, predict(G).reshape(gx.shape), alpha=0.2, cmap='coolwarm', levels=[-2, 0, 2])
    if decision is not None:
        ax.contour(gx, gy, decision(G).reshape(gx.shape), levels=list(levels), colors='k',
                   linestyles=['--', '-', '--'], linewidths=0.8)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', s=14, edgecolor='k', linewidth=0.3)
    ax.set_title(title, fontsize=9)

## 1. Worked example from Math/math.md
Points P1..P6. The doc proposes w=(1,1), b=-4 and margin 2/||w|| = sqrt(2).
We verify the constraints and then let our SMO solver (large C ~ hard margin) find the true optimum.

In [ ]:
P = np.array([[2, 3], [3, 3], [3, 4], [1, 1], [2, 1], [1, 0]], float)
yP = np.array([1, 1, 1, -1, -1, -1], float)
w0, b0 = np.array([1.0, 1.0]), -4.0
print("doc hyperplane: y*(w.x+b) =", yP * (P @ w0 + b0))
print("doc margin 2/||w|| = %.3f" % (2 / np.linalg.norm(w0)))
Q = np.array([2.5, 3.0]); print("f(Q)=%.2f -> class %+d (doc: +1)" % (Q @ w0 + b0, np.sign(Q @ w0 + b0)))

## 2. Linear SVM: hinge loss + L2, subgradient descent
L(w,b) = (lambda/2)||w||^2 + (1/n) sum max(0, 1 - y_i (w.x_i + b)).
Subgradient: lambda*w - y_i x_i for margin-violating points, lambda*w otherwise.
We use full-batch subgradient with a decaying step size (fast in NumPy).

In [ ]:
class LinearSVM:
    def __init__(self, lr=0.1, lambda_param=0.01, n_epochs=500):
        self.lr, self.lam, self.n_epochs = lr, lambda_param, n_epochs

    def decision_function(self, X):
        return X @ self.w + self.b

    def fit(self, X, y):
        n, d = X.shape
        self.w, self.b = np.zeros(d), 0.0
        self.loss_history = []
        best = (np.inf, None, None)
        for t in range(1, self.n_epochs + 1):
            margins = y * (X @ self.w + self.b)
            viol = margins < 1
            gw = self.lam * self.w - (y[viol, None] * X[viol]).sum(0) / n
            gb = -y[viol].sum() / n
            step = self.lr / np.sqrt(t)
            self.w -= step * gw; self.b -= step * gb
            loss = 0.5 * self.lam * self.w @ self.w + np.maximum(0, 1 - y * (X @ self.w + self.b)).mean()
            self.loss_history.append(loss)
            if loss < best[0]: best = (loss, self.w.copy(), self.b)
        _, self.w, self.b = best
        return self

    def predict(self, X):
        return np.where(self.decision_function(X) >= 0, 1, -1)

    @property
    def margin_width(self):
        return 2 / np.linalg.norm(self.w)

# separable data + support vectors (exercise 1)
Xs = np.vstack([rng.normal([-2, -2], 0.7, (40, 2)), rng.normal([2, 2], 0.7, (40, 2))])
ys = np.array([-1] * 40 + [1] * 40)
lin = LinearSVM(lr=0.5, lambda_param=1e-3, n_epochs=3000).fit(Xs, ys)
m = ys * lin.decision_function(Xs)
sv = np.where(m < 1 + 0.1)[0]
print(f"train acc={np.mean(lin.predict(Xs)==ys):.3f}, margin width={lin.margin_width:.2f}, #support vectors (y f(x)<=1.1)={len(sv)}")
dist = np.abs(lin.decision_function(Xs)) / np.linalg.norm(lin.w)
print("closest 5 points to boundary are SVs:", set(np.argsort(dist)[:5]) <= set(sv))
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
plot_boundary(axs[0], lin.predict, Xs, ys, f"LinearSVM, margin={lin.margin_width:.2f}", lin.decision_function)
axs[0].scatter(Xs[sv, 0], Xs[sv, 1], s=90, facecolors='none', edgecolors='g', linewidth=1.5)
axs[1].plot(lin.loss_history); axs[1].set_title('objective vs epoch'); axs[1].set_yscale('log')
fig.tight_layout(); fig.savefig('plots/01_linear_svm.png', dpi=120); plt.close(fig)

## 3. Effect of C on noisy data (exercise 2)
In the primal form lambda = 1/(C n): large C = small lambda = few violations tolerated.

In [ ]:
Xn = np.vstack([rng.normal([-1, -1], 1.2, (100, 2)), rng.normal([1, 1], 1.2, (100, 2))])
yn = np.array([-1] * 100 + [1] * 100)
Xn, _, _ = standardize(Xn)
Xtr, Xte, ytr, yte = split(Xn, yn)
fig, axs = plt.subplots(1, 4, figsize=(16, 3.6))
for ax, C in zip(axs, [0.001, 0.1, 10, 1000]):
    lam = 1.0 / (C * len(ytr))
    s = LinearSVM(lr=0.5, lambda_param=lam, n_epochs=1500).fit(Xtr, ytr)
    plot_boundary(ax, s.predict, Xtr, ytr, f"C={C}: margin={s.margin_width:.2f}, test={np.mean(s.predict(Xte)==yte):.2f}", s.decision_function)
    print(f"C={C:7}: margin width={s.margin_width:8.2f}  train={np.mean(s.predict(Xtr)==ytr):.3f} test={np.mean(s.predict(Xte)==yte):.3f}")
fig.tight_layout(); fig.savefig('plots/02_effect_of_C.png', dpi=110); plt.close(fig)

## 4. Kernels and a simplified SMO solver for the dual
Dual: max sum(a) - 1/2 sum_ij a_i a_j y_i y_j K_ij, s.t. 0<=a_i<=C, sum a_i y_i = 0.
Simplified SMO (Platt / CS229 variant): pick a KKT-violating alpha_i, pair it with a random
alpha_j, solve the 2-variable subproblem analytically, clip to the box, update b.

In [ ]:
def linear_kernel(A, B):            return A @ B.T
def poly_kernel(A, B, c=1.0, d=3):  return (A @ B.T + c) ** d
def rbf_kernel(A, B, gamma=1.0):
    sq = (A**2).sum(1)[:, None] + (B**2).sum(1)[None, :] - 2 * A @ B.T
    return np.exp(-gamma * np.maximum(sq, 0))

class KernelSVM:
    def __init__(self, kernel=rbf_kernel, C=1.0, tol=1e-3, max_passes=10, max_iter=2000, seed=0):
        self.kernel, self.C, self.tol = kernel, C, tol
        self.max_passes, self.max_iter, self.seed = max_passes, max_iter, seed

    def fit(self, X, y):
        r = np.random.default_rng(self.seed)
        n = len(y); K = self.kernel(X, X)
        a, b = np.zeros(n), 0.0
        passes, it = 0, 0
        f = lambda i: (a * y) @ K[:, i] + b
        while passes < self.max_passes and it < self.max_iter:
            changed = 0
            for i in range(n):
                Ei = f(i) - y[i]
                if (y[i] * Ei < -self.tol and a[i] < self.C) or (y[i] * Ei > self.tol and a[i] > 0):
                    j = int(r.integers(0, n - 1)); j = j + 1 if j >= i else j
                    Ej = f(j) - y[j]
                    ai, aj = a[i], a[j]
                    if y[i] != y[j]:
                        L, H = max(0, aj - ai), min(self.C, self.C + aj - ai)
                    else:
                        L, H = max(0, ai + aj - self.C), min(self.C, ai + aj)
                    if L == H: continue
                    eta = 2 * K[i, j] - K[i, i] - K[j, j]
                    if eta >= 0: continue
                    a[j] = np.clip(aj - y[j] * (Ei - Ej) / eta, L, H)
                    if abs(a[j] - aj) < 1e-5: continue
                    a[i] = ai + y[i] * y[j] * (aj - a[j])
                    b1 = b - Ei - y[i]*(a[i]-ai)*K[i, i] - y[j]*(a[j]-aj)*K[i, j]
                    b2 = b - Ej - y[i]*(a[i]-ai)*K[i, j] - y[j]*(a[j]-aj)*K[j, j]
                    b = b1 if 0 < a[i] < self.C else b2 if 0 < a[j] < self.C else (b1 + b2) / 2
                    changed += 1
            passes = passes + 1 if changed == 0 else 0
            it += 1
        sv = a > 1e-6
        self.alpha, self.b = a[sv], b
        self.sv_X, self.sv_y = X[sv], y[sv]
        self.sv_idx = np.where(sv)[0]
        return self

    def decision_function(self, X):
        return self.kernel(X, self.sv_X) @ (self.alpha * self.sv_y) + self.b

    def predict(self, X):
        return np.where(self.decision_function(X) >= 0, 1, -1)

# Solve the math.md example with (almost) hard margin
hm = KernelSVM(kernel=linear_kernel, C=1e3, max_passes=50, max_iter=5000).fit(P, yP)
w_hat = (hm.alpha * hm.sv_y) @ hm.sv_X
print("SMO hard-margin: w=%s b=%.3f margin=%.3f, support vectors=%s"
      % (np.round(w_hat, 3), hm.b, 2 / np.linalg.norm(w_hat), [f"P{i+1}" for i in hm.sv_idx]))
print("(doc's w=(1,1),b=-4 is feasible but its margin sqrt2=1.41 is <= the true optimum margin)")

## 5. Kernel trick on circular data (exercise 3) and gamma effect

In [ ]:
def make_circles(n=300, seed=1):
    r = np.random.default_rng(seed)
    ang = r.uniform(0, 2*np.pi, n); rad = np.where(np.arange(n) < n//2, 1.0, 2.5) + r.normal(0, 0.25, n)
    X = np.c_[rad * np.cos(ang), rad * np.sin(ang)]
    y = np.where(np.arange(n) < n//2, -1, 1)
    return X, y

Xc, yc = make_circles(240)
Xtr, Xte, ytr, yte = split(Xc, yc)
lin_c = LinearSVM(lr=0.5, lambda_param=0.001, n_epochs=800).fit(Xtr, ytr)
print(f"linear SVM on circles: test acc={np.mean(lin_c.predict(Xte)==yte):.3f}")
kernels = {"linear": linear_kernel, "poly d=2": lambda A, B: poly_kernel(A, B, 1, 2),
           "poly d=3": lambda A, B: poly_kernel(A, B, 1, 3), "rbf g=0.5": lambda A, B: rbf_kernel(A, B, 0.5)}
fig, axs = plt.subplots(1, 4, figsize=(16, 3.8))
for ax, (nm, k) in zip(axs, kernels.items()):
    s = KernelSVM(k, C=1.0).fit(Xtr, ytr)
    acc = np.mean(s.predict(Xte) == yte)
    plot_boundary(ax, s.predict, Xtr, ytr, f"{nm}: test={acc:.2f}, #SV={len(s.alpha)}", s.decision_function)
    ax.scatter(s.sv_X[:, 0], s.sv_X[:, 1], s=60, facecolors='none', edgecolors='g')
    print(f"{nm:10s} test acc={acc:.3f}  #SV={len(s.alpha)}/{len(ytr)}")
fig.tight_layout(); fig.savefig('plots/03_kernels_circles.png', dpi=110); plt.close(fig)

# explicit-feature view: RBF Gram matrix sorted by class shows block structure; and phi(x)=x1^2+x2^2 lifts to separable
K = rbf_kernel(Xtr[np.argsort(ytr)], Xtr[np.argsort(ytr)], 0.5)
z = (Xtr ** 2).sum(1)
fig, axs = plt.subplots(1, 2, figsize=(9, 3.8))
im = axs[0].imshow(K, cmap='viridis'); axs[0].set_title('RBF Gram matrix (sorted by class)'); fig.colorbar(im, ax=axs[0])
axs[1].scatter(Xtr[:, 0], z, c=ytr, cmap='coolwarm', s=10); axs[1].set_xlabel('x1'); axs[1].set_ylabel('x1^2+x2^2 (lifted)')
axs[1].set_title('after lifting: a horizontal line separates'); fig.tight_layout()
fig.savefig('plots/04_gram_and_lift.png', dpi=110); plt.close(fig)

fig, axs = plt.subplots(1, 4, figsize=(16, 3.6))
Xg = np.vstack([rng.normal([-1, -1], 1.0, (80, 2)), rng.normal([1, 1], 1.0, (80, 2))]); yg = np.array([-1]*80 + [1]*80)
Xg_tr, Xg_te, yg_tr, yg_te = split(Xg, yg, seed=2)
for ax, g in zip(axs, [0.01, 0.3, 3, 30]):
    s = KernelSVM(lambda A, B, g=g: rbf_kernel(A, B, g), C=1.0).fit(Xg_tr, yg_tr)
    plot_boundary(ax, s.predict, Xg_tr, yg_tr, f"gamma={g}: train={np.mean(s.predict(Xg_tr)==yg_tr):.2f} test={np.mean(s.predict(Xg_te)==yg_te):.2f}", s.decision_function)
fig.tight_layout(); fig.savefig('plots/05_gamma.png', dpi=110); plt.close(fig)

## 6. Hinge vs logistic loss (exercise 4)
Hinge loss is exactly zero for points beyond the margin -> sparse; logistic loss never is.

In [ ]:
def train_logreg(X, y01, lr=0.5, epochs=2000, lam=1e-3):
    w, b = np.zeros(X.shape[1]), 0.0
    for _ in range(epochs):
        p = 1 / (1 + np.exp(-(X @ w + b)))
        w -= lr * ((X.T @ (p - y01)) / len(y01) + lam * w); b -= lr * np.mean(p - y01)
    return w, b

Xh = np.vstack([rng.normal([-1, -1], 1.0, (100, 2)), rng.normal([1, 1], 1.0, (100, 2))]); yh = np.array([-1]*100 + [1]*100)
svm_h = LinearSVM(lr=0.5, lambda_param=0.01, n_epochs=1500).fit(Xh, yh)
w_l, b_l = train_logreg(Xh, (yh + 1) / 2)
n_hinge = int(np.sum(yh * svm_h.decision_function(Xh) < 1))
grad_mag = np.abs(1 / (1 + np.exp(yh * (Xh @ w_l + b_l))))     # |dL/df| for logistic per point
print(f"hinge: {n_hinge}/{len(yh)} points have nonzero gradient")
print(f"logistic: {int(np.sum(grad_mag > 1e-12))}/{len(yh)} nonzero, {int(np.sum(grad_mag > 0.05))} with |grad|>0.05")
u = np.linspace(-2, 3, 200)
fig, ax = plt.subplots(figsize=(5.5, 3.6))
ax.plot(u, np.maximum(0, 1 - u), label='hinge'); ax.plot(u, np.log1p(np.exp(-u)), label='logistic'); ax.plot(u, (u < 0).astype(float), 'k:', label='0-1')
ax.set_xlabel('y f(x)'); ax.legend(); fig.tight_layout(); fig.savefig('plots/06_losses.png', dpi=120); plt.close(fig)

## 7. Support Vector Regression (exercise 5)
Epsilon-insensitive loss max(0, |y - f(x)| - eps). Solved in the primal by subgradient descent
on polynomial features of x (linear SVR in a lifted space). Points outside the tube are support vectors.

In [ ]:
class LinearSVR:
    def __init__(self, eps=0.2, C=10.0, lr=0.05, n_epochs=4000):
        self.eps, self.C, self.lr, self.n_epochs = eps, C, lr, n_epochs
    def fit(self, X, y):
        n, d = X.shape; self.w, self.b = np.zeros(d), 0.0
        for t in range(1, self.n_epochs + 1):
            r = X @ self.w + self.b - y
            out = np.abs(r) > self.eps
            g = np.sign(r) * out
            gw = self.w / (self.C * n) + X.T @ g / n
            self.w -= self.lr / np.sqrt(t) ** 0.5 * gw; self.b -= self.lr / np.sqrt(t) ** 0.5 * g.mean()
        return self
    def predict(self, X): return X @ self.w + self.b

xr = np.sort(rng.uniform(-3, 3, 100)); yr = np.sin(xr) + rng.normal(0, 0.1, 100)
feat = lambda x: np.stack([x ** k for k in range(1, 8)], 1) / np.array([3 ** k for k in range(1, 8)])
svr = LinearSVR(eps=0.2, C=100.0).fit(feat(xr), yr)
pred = svr.predict(feat(xr)); sv_mask = np.abs(pred - yr) >= 0.2 - 1e-2
xg = np.linspace(-3, 3, 300); pg = svr.predict(feat(xg))
print(f"SVR: {sv_mask.sum()}/{len(xr)} points on/outside the eps-tube (support vectors), MSE={np.mean((pred-yr)**2):.4f}")
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.scatter(xr, yr, s=12, c='gray'); ax.scatter(xr[sv_mask], yr[sv_mask], s=60, facecolors='none', edgecolors='g', label='support vectors')
ax.plot(xg, pg, 'r', label='SVR'); ax.fill_between(xg, pg - 0.2, pg + 0.2, alpha=0.2, color='r', label='eps tube'); ax.legend()
fig.tight_layout(); fig.savefig('plots/07_svr.png', dpi=120); plt.close(fig)